# Lecture 2 (optional): Arabica or Robusta? PCA on infrared spectra of coffee

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Edinburgh-Chemistry-Teaching/AdvancedPhysicalChemistry/blob/main/L2_unsupervised/L2_unsupervised_coffee.ipynb)

<a rel="license" href="https://creativecommons.org/licenses/by-sa/4.0/"><img alt="Creative Commons Licence" style="border-width:0" src="https://licensebuttons.net/l/by-sa/4.0/88x31.png" title='This work is licensed under a Creative Commons Attribution-ShareAlike 4.0 International License.' align="right"/></a>

Author: Dr Antonia Mey
Email: antonia.mey@ed.ac.uk

© Antonia Mey, University of Edinburgh, 2026. Text CC BY-SA 4.0; code cells MIT (see `LICENSE` and `LICENSE-CODE` in the repository).

**Time needed:** about 30 minutes. This notebook is **optional** practice for Lecture 2 (unsupervised learning).

### Learning objectives
By the end of this notebook, you should be able to
* Write a set of spectra as a data matrix of samples × features.
* Carry out PCA with NumPy (centre, covariance matrix, eigenvalues and eigenvectors, projection) and read a scree plot, a score plot and a loadings plot.
* Explain why spectra are preprocessed (baseline correction, normalisation) before PCA.
* Make a t-SNE and a UMAP map, and say what they can and cannot add to PCA.

### Table of Contents

0. [The data: 56 infrared spectra](#0-data)
1. [PCA by hand on the raw spectra](#1-pca-raw)
2. [Preprocessing, then PCA again](#2-pca-prep)
3. [t-SNE and UMAP](#3-tsne-umap)
4. [Key points](#4-key-points)

### The data

Coffee comes from two species: *Arabica* (milder, more expensive) and *Robusta* (more bitter, more caffeine, cheaper). Once the beans are roasted, ground and made into instant coffee, you cannot tell them apart by eye, so food authentication labs look for a chemical fingerprint instead.

We use 56 mid-infrared spectra of freeze-dried instant coffee (29 Arabica, 27 Robusta), measured by diffuse reflectance FTIR (DRIFT) between 810 and 1910 cm⁻¹ at the Institute of Food Research, Norwich (now the Quadram Institute).

This data set is also used in the Year 3 physical chemistry laboratory on PCA, where it is analysed with the Orange Data Mining program. If you have done that lab you will recognise the plots; here we do the analysis in code and look at what happens inside PCA.

**The idea of this notebook:** we pretend we do *not* know which species each sample is. Can PCA find the two species from the spectra alone? We only use the labels to check.

*Data source:* Quadram Institute Bioscience, [example data sets](https://csr.quadram.ac.uk/example-datasets-for-download/); R. Briandet, E. K. Kemsley and R. H. Wilson, *J. Agric. Food Chem.* **44**, 170–174 (1996), [doi:10.1021/jf950305a](https://doi.org/10.1021/jf950305a); G. Downey, R. Briandet, R. H. Wilson and E. K. Kemsley, *J. Agric. Food Chem.* **45**, 4357–4361 (1997).

### How to open this notebook

* **Noteable (recommended):** open it from the Lecture 2 page on Learn. Everything you need is already installed.
* **Google Colab (backup):** click the *Open in Colab* button at the top. You need a Google account. Save a copy to your Drive (*File → Save a copy in Drive*) or your answers will be lost.
* **Your own computer:** you need `numpy`, `pandas`, `matplotlib`, `scikit-learn` and `umap-learn` (see `requirements.txt` in the repository).

**<span style="color:black">Jupyter Cheat Sheet</span>**
- To run the currently highlighted cell and move focus to the next cell, hold <kbd>&#x21E7; Shift</kbd> and press <kbd>&#x23ce; Enter</kbd>;
- To get help for a specific function, place the cursor within the function's brackets, hold <kbd>&#x21E7; Shift</kbd>, and press <kbd>&#x21E5; Tab</kbd>;
- Cells marked `# FIXME` are for you to complete.

# Import libraries

In [ ]:
# UMAP is not part of scikit-learn: install it if it is missing (e.g. on Colab)
try:
    import umap
except ImportError:
    %pip install -q umap-learn
    import umap

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

warnings.filterwarnings("ignore", message=".*n_jobs.*")   # UMAP: harmless note about random_state

BLUE, ORANGE = "#2a78d6", "#eb6834"
SPECIES_COLOURS = {"Arabica": BLUE, "Robusta": ORANGE}

# The data file lives in the data/ folder of the course repository.
# If it is not found locally (e.g. on Colab), it is downloaded from GitHub.
DATA_URL = "https://raw.githubusercontent.com/Edinburgh-Chemistry-Teaching/AdvancedPhysicalChemistry/main/data/"


def load(name):
    try:
        return pd.read_csv("../data/" + name)
    except FileNotFoundError:
        return pd.read_csv(DATA_URL + name)


def plot_by_species(Y, ax, title=""):
    """Scatter plot of the first two columns of Y, coloured by species."""
    for s, colour in SPECIES_COLOURS.items():
        m = species == s
        ax.scatter(Y[m, 0], Y[m, 1], s=25, color=colour, label=s)
    ax.set_title(title)
    ax.legend()

# 0. The data: 56 infrared spectra <a id="0-data"></a>

Each row of the file is one **sample** (one coffee). The columns are the intensities at 286 wavenumbers, so each wavenumber is one **feature**. Together they form the **data matrix** $X$ with $N = 56$ rows and $d = 286$ columns. The species is a **label**: we keep it separate and only use it to check our results.

In [ ]:
df = load("coffee_ftir_spectra.csv")
species = df["species"].values                     # label (kept aside!)
X = df.drop(columns=["sample", "species"]).values  # data matrix: one spectrum per row
wavenumber = df.columns[2:].astype(float).values   # / cm^-1

print("data matrix X:", X.shape, "(N samples, d features)")
print(df["species"].value_counts())

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(wavenumber, X.T, color="grey", lw=0.5)
ax.set_xlabel("wavenumber / cm$^{-1}$"); ax.set_ylabel("DRIFT intensity")
ax.invert_xaxis()                                  # IR convention: high wavenumber on the left
ax.set_title("56 coffee spectra, no labels")
plt.show()

The spectra all have the same shape but very different overall heights. In diffuse reflectance the height depends on things such as how finely the powder is ground and how it is packed in the sample cup, not only on the chemistry. Keep this in mind for Section 1.

# 1. PCA by hand on the raw spectra <a id="1-pca-raw"></a>

These are exactly the steps from the board in Lecture 2, now for 286 features instead of 2:

1. **Centre** the data: subtract the mean spectrum from every spectrum.
2. Compute the **covariance matrix** $\mathbf C = \frac{1}{N}\mathbf X_c^\mathsf{T}\mathbf X_c$ (a $286 \times 286$ matrix).
3. Find its **eigenvalues** $\lambda_k$ and **eigenvectors** $\mathbf v_k$, sorted from the largest eigenvalue down. The eigenvectors are the principal components (PCs); each eigenvalue is the variance along its PC.
4. **Fraction of variance explained** by PC $k$: $\lambda_k / \sum_j \lambda_j$.
5. **Project** each spectrum onto the PCs: the *scores* are $\mathbf X_c \mathbf V$.

We do **not** standardise each wavenumber here: all features are intensities in the same units, and dividing by the standard deviation would blow up the flat, noisy parts of the spectrum.

`np.linalg.eigh` is the NumPy function for eigenvalues of a symmetric matrix; it returns them in *ascending* order.

In [ ]:
N = X.shape[0]
Xc = X - X.mean(axis=0)               # 1. centre
C = Xc.T @ Xc / N                     # 2. covariance matrix
eigvals, eigvecs = np.linalg.eigh(C)  # 3. ascending order
order = np.argsort(eigvals)[::-1]     #    largest first
eigvals, eigvecs = eigvals[order], eigvecs[:, order]

scores = Xc @ eigvecs                 # 5. projection onto all PCs
print("C has shape", C.shape)

## Tasks 1 <a id="tasks-1"></a>
1. Compute the fraction of variance explained by each PC and the cumulative fraction, and plot both for the first 10 PCs (a *scree plot*). How much does PC1 explain?
2. Make a score plot of PC1 against PC2, coloured by species with `plot_by_species(scores, ax)`. Does PC1 separate Arabica from Robusta?
3. Compute the total intensity of each spectrum, `X.sum(axis=1)`, and plot the PC1 score against it. What is PC1 measuring?
4. Make a score plot of PC2 against PC3 (`scores[:, 1:3]`). Where has the species information gone?

In [ ]:
# 1. Scree plot
# FIXME

In [ ]:
# 2. Score plot PC1 vs PC2
# FIXME

In [ ]:
# 3. PC1 against total intensity
# FIXME

In [ ]:
# 4. Score plot PC2 vs PC3
# FIXME

*Your answers to questions 1–4 here.*

# 2. Preprocessing, then PCA again <a id="2-pca-prep"></a>

We remove the variation we are not interested in before doing PCA:

* **Baseline correction:** subtract the straight line through the first and last point of each spectrum.
* **Area normalisation:** divide each spectrum by its area, so every spectrum has area 1. Only the *shape* of the spectrum is left, not its height.

(The whisky spectra in the Year 3 lab are treated in a similar way, with SNV: each spectrum minus its own mean, divided by its own standard deviation.)

In [ ]:
# straight line through the first and last point of each spectrum
baseline = X[:, [0]] + (X[:, [-1]] - X[:, [0]]) * (wavenumber - wavenumber[0]) / (wavenumber[-1] - wavenumber[0])
X_base = X - baseline
X_prep = X_base / np.abs(np.trapezoid(X_base, wavenumber, axis=1))[:, None]   # area 1

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(wavenumber, X_prep.T, color="grey", lw=0.5)
axes[0].set_title("preprocessed spectra, no labels")
for s, colour in SPECIES_COLOURS.items():
    axes[1].plot(wavenumber, X_prep[species == s].mean(axis=0), color=colour, label=f"{s} (mean)")
axes[1].legend(); axes[1].set_title("mean spectrum of each species")
for ax in axes:
    ax.set_xlabel("wavenumber / cm$^{-1}$"); ax.invert_xaxis()
plt.show()

The two species look very alike. The differences are small, for example around 1650 and 1706 cm⁻¹ (caffeine and chlorogenic acid, higher in Robusta) and 1760 cm⁻¹ (C=O stretch of the fats and lipids, higher in Arabica).

This time we let `scikit-learn` do the PCA. `PCA` centres the data for us; `fit_transform` returns the scores, `explained_variance_ratio_` the fractions of variance and `components_` the eigenvectors (one PC per **row**).

In [ ]:
pca = PCA(n_components=5)
scores_prep = pca.fit_transform(X_prep)
print("fraction of variance:", np.round(pca.explained_variance_ratio_, 3))

## Tasks 2 <a id="tasks-2"></a>
1. Make the score plot of PC1 against PC2 for the preprocessed spectra, coloured by species. Which PC separates Arabica from Robusta now?
2. The **loadings** of a PC are the entries of its eigenvector: how much each wavenumber contributes to it. For spectra it is easiest to plot them as a spectrum. Plot the loadings of the PC that separates the species against wavenumber, and mark 1650, 1706 and 1760 cm⁻¹ with `ax.axvline`.
3. Do the peaks at 1650 and 1706 cm⁻¹ have the same sign as the peak at 1760 cm⁻¹? Which species lies on the side of positive scores, and does that agree with the chemistry above?

In [ ]:
# 1. Score plot of the preprocessed spectra
# FIXME

In [ ]:
# 2. Loadings of PC2 as a spectrum
# FIXME

*Your answers to questions 1 and 3 here.*

# 3. t-SNE and UMAP <a id="3-tsne-umap"></a>

PCA is **linear**: it can only rotate the data and drop directions. t-SNE and UMAP (Lecture 2) are **non-linear** maps that try to keep each sample's **neighbours** together in 2D. In t-SNE the *perplexity* sets roughly how many neighbours count; in UMAP this is `n_neighbors`.

Remember from the lecture: in a t-SNE or UMAP map the sizes of the groups, the gaps between them and the axes have no meaning. Only *which samples are close together* does. A good, interactive read: [How to use t-SNE effectively](https://distill.pub/2016/misread-tsne/) (distill.pub).

In [ ]:
Y_tsne = TSNE(perplexity=15, random_state=0).fit_transform(X_prep)
Y_umap = umap.UMAP(n_neighbors=15, random_state=0).fit_transform(X_prep)

## Tasks 3 <a id="tasks-3"></a>
1. Plot the PCA score plot (`scores_prep`), the t-SNE map and the UMAP map side by side, coloured by species. Do t-SNE and UMAP separate the species better than PCA, and by how much?
2. Now run t-SNE and UMAP (same settings) on the **raw** spectra `X` and plot them. What happens?
3. A colleague says: "the preprocessing does not matter, t-SNE will find the groups anyway". What would you reply?

In [ ]:
# 1. PCA, t-SNE and UMAP side by side
# FIXME

In [ ]:
# 2. t-SNE and UMAP on the raw spectra
# FIXME

*Your answers to questions 1–3 here.*

# 4. Key points <a id="4-key-points"></a>

* Unsupervised learning works on the data matrix $X$ alone; labels, if we have them, are only for checking.
* **PCA** = eigenvectors of the covariance matrix; each eigenvalue is the variance along its PC. Score plot = samples projected onto the PCs; loadings = the eigenvector entries, which for spectra can be read like a spectrum.
* PCA finds the direction of **largest variance**, and that need not be the chemistry: here, in the raw spectra, it was the overall intensity (93 %).
* **Preprocessing** (baseline correction, normalisation, or scaling of features) decides what "variance" and "distance" mean. Choose it with the measurement in mind.
* **t-SNE** and **UMAP** keep neighbours together. They can help when the groups are not separated along straight lines, but they cannot fix badly chosen features, and group sizes, gaps and axes in their maps mean little.

**Questions to think about:**
1. You add 10 % water/ethanol blanks to a set of whisky UV-vis spectra. Where would you expect them on a PCA score plot, and why?
2. Why might area normalisation be a bad idea if you wanted to measure the *total* caffeine content rather than tell the species apart?